In [36]:
import numpy as np
import matplotlib.pyplot as plt
#write a code to 

In [37]:
import os
import numpy as np

dir = ['/home/apa2237/generative_model_work/adalead/aav/generative_results',\
       '/home/apa2237/generative_model_work/dyna_ppo/FLEXS/examples/results_aav',
    #    '/home/apa2237/generative_model_work/gflownet/BioSeq-GFN-AL/result_aav',
       ]
# Find all .npy files in the directories that contain dicts
# ignore the files that start with 'sequences_'
# do not plot ourGREEDY results
npy_files = []
method_names = []
for d in dir:
    for file in os.listdir(d):
        if file.endswith('.npy') and not file.startswith(('sequences_', 'cbas', 'dbas', 'bo','dynappo')) and 'ourGREEDY' not in file:
            npy_files.append(os.path.join(d, file))
            method_names.append(os.path.basename(file).split('_')[0])

# reshuffle to make "Ours" last if present
if 'Ours' in method_names:
    idx = method_names.index('Ours')
    npy_files.append(npy_files.pop(idx))
    method_names.append(method_names.pop(idx))
print(npy_files)
print(method_names)

# in the code below, replace f'File {i+1}' with method_names[i]

# Load all dicts from the found .npy files
dicts = []
for file in npy_files:
    arr = np.load(file, allow_pickle=True)
    if isinstance(arr.item(), dict):
        dicts.append(arr.item())

common_keys = set.intersection(*(set(d.keys()) for d in dicts))

# Dictionary to store vals for each method and key
method_vals_storage = {key: {} for key in common_keys}
key_to_minmax = {}
for key in common_keys:
    max_values = []  # Store max value for each method
    
    for i, dct in enumerate(dicts):
        vals = np.array(dct[key])
        
        if len(vals.shape) == 3:
            vals = vals[:, 0:10, 2]
            vals = np.mean(vals, axis=0)
        else:
            vals = np.mean(vals, axis=0)
        
        # Store vals for this method
        method_vals_storage[key][method_names[i]] = vals
        
        # Get max value for this method
        max_val = np.max(vals)
        max_values.append(max_val)
    
    # Find minimum of the max values
    min_max_value = np.min(max_values)
    min_max_method_idx = np.argmin(max_values)
    min_max_method = method_names[min_max_method_idx]
    key_to_minmax[key] = min_max_value ## this is what we need 
    
print(key_to_minmax)

['/home/apa2237/generative_model_work/adalead/aav/generative_results/adalead.npy', '/home/apa2237/generative_model_work/adalead/aav/generative_results/cmaes.npy', '/home/apa2237/generative_model_work/adalead/aav/generative_results/our_1.npy', '/home/apa2237/generative_model_work/adalead/aav/generative_results/ourEXPLOIT_1.npy']
['adalead.npy', 'cmaes.npy', 'our', 'ourEXPLOIT']
{50: 0.3819619916081428, 100: 0.38788630901277066, 20: 0.34716580212116244}


## Preparing the data for metric calculation

In [38]:

top_per = 30

data_path = "/home/apa2237/generative_model_work/datasets/tf_ARX_REF_R1/"

seq_start = np.load(f'{data_path}/seq_test.npy', allow_pickle=True).reshape((-1,1))
y_start = np.load(f'{data_path}/y_test.npy', allow_pickle=True).reshape((-1,1))

print('Before filtering',len(seq_start), len(y_start))
print('Mean before', np.mean(y_start))
print('Min before', np.min(y_start))
print('Max before', np.max(y_start))

cap = max(y_start)
floor = min(y_start)
cutoff = 0.2
print('Cutoff', cutoff)
below_idx = (y_start<cutoff)
# print('Below_idx', np.sum(below_idx*1))
print(below_idx.shape,seq_start.shape)

seq_start = seq_start[below_idx]
y_start = y_start[below_idx]

print('Filtering==================')
print('After filtering',len(seq_start), len(y_start))
print('Mean after', np.mean(y_start))
print('Max after', np.max(y_start))

# print(seq_start)


Before filtering 8356 8356
Mean before 0.4462841864646173
Min before 0.0
Max before 0.997026473644679
Cutoff 0.2
(8356, 1) (8356, 1)
Filtering==================
After filtering 353 353
Mean after 0.16152546562691034
Max after 0.1999567861221718


In [39]:
from polyleven import levenshtein

def edit_dist(seq1, seq2):
    return levenshtein(seq1, seq2) / 1

import itertools

def mean_pairwise_distances(args, seqs):
    dists = []
    for pair in itertools.combinations(seqs, 2):
        dists.append(edit_dist(*pair))
    return np.mean(dists)

def novelty(list_1, list_2):
    dists = []
    for s1, s2 in itertools.product(list_1, list_2):  # cross pairs only
        # print(s1,s2)
        dists.append(edit_dist(s1, s2))
    return np.mean(dists) if dists else 0.0


In [40]:
def auc_cal(prop):
    for k in prop.keys():  # show up to 3 keys
        if k not in [20, 50, 100]:
            continue
        # Get data
        if len(prop[k].shape) == 2:
            temp = prop[k]
        else:
            temp = prop[k][..., 2]
        area = []
        for j in range(temp.shape[0]):
            y = temp[j,:]
            # shifting y so that min is zero
            # y = y - np.min(y)
            x = np.arange(len(y))
            auc = np.trapz(y, x)
            area.append(auc)
        print(f"Samples:{k}, AUC:{np.mean(area):.2f} ± {np.std(area):.2f}")
    # return area

In [41]:
def find_nearest(arr, target):
    L,R = 0, len(arr)-1
    while L<R:
        mid = (L+R)//2
        if arr[mid]<target:
            L = mid+1
        else:
            R = mid
        
    return R

In [42]:
def cal_d_n(sequences,prop_ada):
    div_res = []
    for k1 in sequences.keys():
        if k1 not in [20, 50, 100]:
            continue
        sum_dist = []
        novel_dist = []
        for k2 in sequences[k1].keys():
            if len(prop_ada[k1][k2].shape)==2:
                idx = find_nearest(prop_ada[k1][k2][1:,2], key_to_minmax[k1])
            else:
                idx = find_nearest(prop_ada[k1][k2][1:], key_to_minmax[k1])
            

            # # if sequences[k1][k2][idx]>=key_to_minmax[k1]:
            # if len(prop_ada[k1][k2].shape)==2:
            #     if 0.8*key_to_minmax[k1] <= prop_ada[k1][k2][idx,2] <= 1.2*key_to_minmax[k1]:
            #         sim = mean_pairwise_distances(None, sequences[k1][k2][idx+1])
            #         sum_dist.append(sim)
            # else:
            #     if 0.8*key_to_minmax[k1] <= prop_ada[k1][k2][idx] <= 1.2*key_to_minmax[k1]:
            sim = mean_pairwise_distances(None, sequences[k1][k2][idx+1])
            sum_dist.append(sim)
            
            nov = novelty(sequences[k1][k2][9],seq_start)
            novel_dist.append(nov)
            
        print(f"Samples:{k1}, Diversity:{np.mean(sum_dist):.2f} ± {np.std(sum_dist):.2f}, \
        Novelty:{np.mean(novel_dist):.2f} ± {np.std(novel_dist):.2f}")
        div_res.append(np.mean(sum_dist))
        
    return div_res


## AUC

In [43]:
data_dir = './generative_results'
path_2 = '/home/apa2237/generative_model_work/dyna_ppo/FLEXS/examples/results_aav'

prop_bo = np.load(f'{path_2}/bo.npy', allow_pickle=True).tolist()
sequences_bo  = np.load(f'{path_2}/sequences_bo.npy', allow_pickle=True).tolist()
print('--------------BO----------------')
auc_cal(prop_bo)

prop_cmaes = np.load(f'{data_dir}/cmaes.npy', allow_pickle=True).tolist()
sequences_cmaes = np.load(f'{data_dir}/sequences_cmaes.npy', allow_pickle=True).tolist()
print('--------------CMA-ES----------------')
# print(prop_cmaes[20].shape)
auc_cal(prop_cmaes)

prop_ada = np.load(f'{data_dir}/adalead.npy', allow_pickle=True).tolist()
sequences_ada = np.load(f'{data_dir}/sequences_adalead.npy', allow_pickle=True).tolist()
print('--------------AdaLead----------------')
print(prop_ada[20].shape)
auc_cal(prop_ada)

# # Dbas, Cbas, DynaPPO
prop_cbas = np.load(f'{path_2}/cbas.npy', allow_pickle=True).tolist()
sequences_cbas  = np.load(f'{path_2}/sequences_cbas.npy', allow_pickle=True).tolist()
print('--------------Cbas----------------')
auc_cal(prop_cbas)

## Dbas, Cbas, DynaPPO
prop_dbas = np.load(f'{path_2}/dbas.npy', allow_pickle=True).tolist()
sequences_dbas = np.load(f'{path_2}/sequences_dbas.npy', allow_pickle=True).tolist()
print('--------------Dbas----------------')
# print(prop_dbas[20].shape)
auc_cal(prop_dbas)

# ## GFlowNet
# path_gfn = '/home/apa2237/generative_model_work/gflownet/BioSeq-GFN-AL/result_tf_arx_l343q'
# prop_gfn = np.load(f'{path_gfn}/gflownet_tf_arx_l343q.npy', allow_pickle=True).tolist()
# sequences_gfn = np.load(f'{path_gfn}/sequences_gflownet_tf_arx_l343q.npy', allow_pickle=True).tolist()
# print('--------------GFlowNet----------------')
# auc_cal(prop_gfn)


# prop_dyna = np.load(f'{path_2}/dynappo.npy', allow_pickle=True).tolist()
# sequences_dyna = np.load(f'{path_2}/sequences_dynappo.npy', allow_pickle=True).tolist()
# print('--------------Dynappo----------------')
# # print(prop_dyna[20].shape)
# auc_cal(prop_dyna)

prop_mode = np.load(f'{data_dir}/our_1.npy', allow_pickle=True).tolist()
sequences_mode = np.load(f'{data_dir}/sequences_our_1.npy', allow_pickle=True).tolist()
print('--------------IDEAS----------------')
auc_cal(prop_mode)


prop_mode_expl = np.load(f'{data_dir}/ourEXPLOIT_1.npy', allow_pickle=True).tolist()
sequences_mode_expl = np.load(f'{data_dir}/sequences_ourEXPLOIT_1.npy', allow_pickle=True).tolist()
print('--------------IDEAS-X----------------')
auc_cal(prop_mode_expl)

# prop_mode = np.load(f'{data_dir}/our_1_motif_1.npy', allow_pickle=True).tolist()
# sequences_mode = np.load(f'{data_dir}/sequences_our_1_motif_1.npy', allow_pickle=True).tolist()
# print('--------------IDEAS_motif_1----------------')
# auc_cal(prop_mode)

--------------BO----------------
Samples:20, AUC:2.77 ± 0.30
Samples:50, AUC:2.35 ± 0.18
Samples:100, AUC:2.22 ± 0.14
--------------CMA-ES----------------
Samples:20, AUC:3.36 ± 0.17
Samples:50, AUC:3.46 ± 0.14
Samples:100, AUC:3.48 ± 0.15
--------------AdaLead----------------
(10, 11, 3)
Samples:20, AUC:3.33 ± 0.07
Samples:50, AUC:3.59 ± 0.14
Samples:100, AUC:3.81 ± 0.15
--------------Cbas----------------
Samples:100, AUC:2.08 ± 0.08
Samples:20, AUC:2.29 ± 0.19
Samples:50, AUC:2.27 ± 0.11
--------------Dbas----------------
Samples:20, AUC:1.60 ± 0.30
Samples:50, AUC:2.16 ± 0.27
Samples:100, AUC:2.11 ± 0.06
--------------IDEAS----------------
Samples:50, AUC:4.22 ± 0.15
Samples:20, AUC:3.90 ± 0.21
Samples:100, AUC:4.34 ± 0.11
--------------IDEAS-X----------------
Samples:50, AUC:3.85 ± 0.21
Samples:20, AUC:3.58 ± 0.14
Samples:100, AUC:4.20 ± 0.22


### AUC Summary Table

| Method    | 20              | 50              | 100             |
|-----------|-----------------|-----------------|-----------------|
| BO        | 2.77 ± 0.30     | 2.35 ± 0.18     | 2.22 ± 0.14     |
| CMA-ES    | 3.36 ± 0.17     | 3.46 ± 0.14     | 3.48 ± 0.15     |
| AdaLead   | 3.33 ± 0.07     | 3.59 ± 0.14     | 3.81 ± 0.15     |
| Cbas      | 2.29 ± 0.19     | 2.27 ± 0.11     | 2.08 ± 0.08     |
| Dbas      | 1.60 ± 0.30     | 2.16 ± 0.27     | 2.11 ± 0.06     |
| IDEAS     | **3.90 ± 0.21** | **4.22 ± 0.15** | **4.34 ± 0.11** |
| IDEAS-X   | 3.58 ± 0.14     | 3.85 ± 0.21     | 4.20 ± 0.22     |

## Diversity and Novelty

In [44]:

print('--------------AdaLead----------------')
cal_d_n(sequences_ada,prop_ada)

print('--------------CMA-ES----------------')
cal_d_n(sequences_cmaes, prop_cmaes)

print('--------------Dbas----------------')
cal_d_n(sequences_dbas, prop_dbas)

print('--------------Cbas----------------')
cal_d_n(sequences_cbas, prop_cbas)

print('--------------IDEAS----------------')
div_idea = cal_d_n(sequences_mode, prop_mode  )
# cal_d_n(sequences_cbas, prop_cbas)

print('--------------BO----------------')
cal_d_n(sequences_bo, prop_bo  )

print('--------------IDEAS-X----------------')
div_idea_expl = cal_d_n(sequences_mode_expl, prop_mode_expl)

# find the mean difference between div_idea and div_idea_expl]
diff_div = [100*(i-j)/j for i,j in zip( div_idea, div_idea_expl)]
print(f'Mean diff : {np.mean(diff_div):.2f}')

--------------AdaLead----------------


Samples:20, Diversity:3.58 ± 0.87,         Novelty:738.00 ± 0.00
Samples:50, Diversity:8.86 ± 2.73,         Novelty:738.00 ± 0.00
Samples:100, Diversity:24.73 ± 8.75,         Novelty:738.00 ± 0.00
--------------CMA-ES----------------
Samples:20, Diversity:418.10 ± 3.03,         Novelty:738.00 ± 0.00
Samples:50, Diversity:426.96 ± 2.26,         Novelty:738.00 ± 0.00
Samples:100, Diversity:431.35 ± 1.19,         Novelty:738.00 ± 0.00
--------------Dbas----------------
Samples:20, Diversity:21.42 ± 21.66,         Novelty:741.00 ± 0.00
Samples:50, Diversity:24.59 ± 21.50,         Novelty:741.00 ± 0.00
Samples:100, Diversity:20.92 ± 15.83,         Novelty:741.00 ± 0.00
--------------Cbas----------------
Samples:100, Diversity:63.28 ± 58.02,         Novelty:741.00 ± 0.00
Samples:20, Diversity:9.01 ± 3.00,         Novelty:741.00 ± 0.00
Samples:50, Diversity:50.10 ± 54.92,         Novelty:741.00 ± 0.00
--------------IDEAS----------------
Samples:50, Diversity:138.37 ± 51.79,         Novelty:73